Error message on Mogonet

```bash
Process `CROSS_VALIDATION:CV_PYTHON:MOGONET:MOGONET_TRAIN (multiomics-sim10-fold_2)` terminated with an error exit status (1)

Command executed:

run_mogonet.py 			--fold_path=fold_2 			--label=multiomics-sim10-fold_2 > 			multiomics-sim10-fold_2-train.log

Command exit status:
1

Command output:
(empty)

Command error:
INFO:    underlay of /etc/localtime required more than 50 (92) bind mounts
/usr/local/lib/python3.10/dist-packages/mogonet/utils.py:79: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:618.)
return sparse_tensortype(indices, values, x.size())
Traceback (most recent call last):
File "/scratch/st-singha53-1/tliang19/multi-omics-pipeline/modules/mogonet/train/resources/usr/bin/run_mogonet.py", line 104, in <module>
  main(
File "/scratch/st-singha53-1/tliang19/multi-omics-pipeline/modules/mogonet/train/resources/usr/bin/run_mogonet.py", line 73, in main
  model_dict, test_input = train_mogonet(
File "/usr/local/lib/python3.10/dist-packages/mogonet/train_mogonet.py", line 114, in train_mogonet
  model_dict = init_model_dict(num_view, num_class, dim_list, dim_he_list, dim_hvcdn)
File "/usr/local/lib/python3.10/dist-packages/mogonet/models.py", line 95, in init_model_dict
  model_dict["E{:}".format(i+1)] = GCN_E(dim_list[i], dim_he_list, gcn_dopout)
File "/usr/local/lib/python3.10/dist-packages/mogonet/models.py", line 41, in __init__
  self.gc3 = GraphConvolution(hgcn_dim[1], hgcn_dim[2])
IndexError: list index out of range

Work dir:
/scratch/st-singha53-1/tliang19/multi-omics-pipeline/work/95/5b3e4901ad0d49e66136e6ebbd39dc
```

In [1]:
# Imports
import os

In [2]:
# workdir of the thing
work_dir = "../work/95/5b3e4901ad0d49e66136e6ebbd39dc/"
fold_path = f"{work_dir}/fold_2"
label = "multiomics-sim10-fold_2"

In [3]:
# Command executed
# run_mogonet.py  --fold_path=fold_2  --label=multiomics-sim10-fold_2 > multiomics-sim10-fold_2-train.log

Then the file content is:

```python
#!/usr/bin/env python

"""
This is a Mogonet script with cli arguments support
Change this script for your use and write more useful doc here

Usage:
  run_mogonet.py [options]

Options:
  -h --help                   Show this message
  --fold_path=FOLD_PATH       Directory containing one split directory of relevant input [default: None] 
  --label=LABEL               Label of dataset and fold iteration [default: empty]
"""

from docopt import docopt
from mogonet.train_mogonet import train_mogonet
from mogonet.utils import save_model_dict
import os
import glob
import pandas as pd
import pickle

# =============================================================================
# Little utilities to use here
# =============================================================================
def get_view_list(data_folder):
    # First list all *_featname.csv in current data folder
    pattern = "_featname.csv"
    csvs = [csv for csv in os.listdir(data_folder) if pattern in csv]
    # Remove the pattern, so left should be block name
    view_list = [csv.replace(pattern, "") for csv in csvs]
    return view_list

def write_metadata(data_folder, label):
    dataset_name  = label.split('-fold')[0]
    metadata_file = glob.glob(os.path.join(data_folder, "*meta*"))[0]
    output_file = f"{dataset_name}-{os.path.basename(metadata_file)}"
    # Then read in and add the dataset_name in
    meta_df = pd.read_csv(metadata_file)
    meta_df["dataset"] = dataset_name
    meta_df.to_csv(output_file, header=True, index=False)
    print(f"Wrote to {output_file}")
    return meta_df



# Train a model network from mogonet, assuming having those right inputs 
# from upstream process
def main(
    data_folder, 
    label,
    view_list=None,
    num_class=2,
    lr_e_pretrain=1e-3,
    lr_e=5e-4,
    lr_c=1e-3,
    num_epoch_pretrain=50,
    num_epoch=200,
    test_interval=50,
    adj_parameter=8
    ):
  # Do something with mogonet here
  # ===========================================================================
  #                         Handle parameter checks
  # ===========================================================================
  # Generate the view list by finding all *_featname.csv of each block
  if view_list is None:
    view_list = get_view_list(data_folder=data_folder)
  # ===========================================================================
  # Main executing goes here
  # ===========================================================================
  model_dict, test_input = train_mogonet(
        data_folder         = data_folder, 
        view_list           = view_list, 
        num_class           = num_class, 
        lr_e_pretrain       = lr_e_pretrain, 
        lr_e                = lr_e, 
        lr_c                = lr_c, 
        num_epoch_pretrain  = num_epoch_pretrain, 
        num_epoch           = num_epoch, 
        adj_parameter       = adj_parameter
  )
  # Parse label and choose output file to write
  model_file = f"{label}-model.pt"
  # Then save the trained model to disk
  save_model_dict(model_dict, model_file)
  # Also write the test input to file so that it is pass to downstream
  test_file = f"{label}-test_input.pkl"
  with open(test_file, 'wb') as f:
    pickle.dump(test_input, f)
  print(f"\nSaved input for testing at: {test_file}")
  write_metadata(data_folder, label)
  # And writing the metadata file for downstream
  
  
  return model_dict
# Execute it here
if __name__ == '__main__':
  # Parse docopt
  args = docopt(__doc__)
  # Execute runner
  #grid_params = {}
  main(
    data_folder = args["--fold_path"],
    label       = args["--label"]
  )
```

In [28]:
def gen_trte_adj_mat(data_tr_list, data_trte_list, trte_idx, adj_parameter):
    adj_metric = "cosine" # cosine distance
    adj_train_list = []
    adj_test_list = []
    for i in range(len(data_tr_list)):
        adj_parameter_adaptive = cal_adj_mat_parameter(adj_parameter, data_tr_list[i], adj_metric)
        adj_train_list.append(gen_adj_mat_tensor(data_tr_list[i], adj_parameter_adaptive, adj_metric))
        adj_test_list.append(gen_test_adj_mat_tensor(data_trte_list[i], trte_idx, adj_parameter_adaptive, adj_metric))
    
    return adj_train_list, adj_test_list

In [6]:
# def get_view_list(data_folder):
#     # First list all *_featname.csv in current data folder
#     pattern = "_featname.csv"
#     csvs = [csv for csv in os.listdir(data_folder) if pattern in csv]
#     # Remove the pattern, so left should be block name
#     view_list = [csv.replace(pattern, "") for csv in csvs]
#     return view_list

def write_metadata(data_folder, label):
    dataset_name  = label.split('-fold')[0]
    metadata_file = glob.glob(os.path.join(data_folder, "*meta*"))[0]
    output_file = f"{dataset_name}-{os.path.basename(metadata_file)}"
    # Then read in and add the dataset_name in
    meta_df = pd.read_csv(metadata_file)
    meta_df["dataset"] = dataset_name
    meta_df.to_csv(output_file, header=True, index=False)
    print(f"Wrote to {output_file}")
    return meta_df

In [7]:
class MogonetInput():
    def __init__(self, data_folder, label, view_list=None, num_class=2, lr_e_pretrain=1e-3, lr_e=5e-4, lr_c=1e-3, num_epoch_pretrain=50, num_epoch=200, adj_parameter=8):
        self.data_folder = data_folder
        self.label = label
        if view_list is None:
            self.view_list = get_view_list(data_folder)
        self.num_class = num_class
        self.lr_e_pretrain = lr_e_pretrain
        self.lr_e=lr_e
        self.lr_c=lr_c
        self.num_epoch_pretrain = num_epoch_pretrain
        self.num_epoch = num_epoch
        self.adj_parameter = 8
        
    def get_view_list(self, data_folder):
        # First list all *_featname.csv in current data folder
        pattern = "_featname.csv"
        csvs = [csv for csv in os.listdir(data_folder) if pattern in csv]
        # Remove the pattern, so left should be block name
        view_list = [csv.replace(pattern, "") for csv in csvs]
        return view_list

In [15]:
kl = MogonetInput(fold_path, label)
m = kl

In [9]:
kl.view_list

['b_block', 'a_block']

In [11]:
from mogonet.train_mogonet import train_mogonet

In [12]:
def main(
    m
    ):
  # Do something with mogonet here
  # ===========================================================================
  #                         Handle parameter checks
  # ===========================================================================
  # Generate the view list by finding all *_featname.csv of each block
  if m.view_list is None:
    m.view_list = get_view_list(data_folder=m.data_folder)
  # ===========================================================================
  # Main executing goes here
  # ===========================================================================
  model_dict, test_input = train_mogonet(
      data_folder         = m.data_folder, 
      view_list           = m.view_list, 
      num_class           = m.num_class, 
      lr_e_pretrain       = m.lr_e_pretrain, 
      lr_e                = m.lr_e, 
      lr_c                = m.lr_c, 
      num_epoch_pretrain  = m.num_epoch_pretrain, 
      num_epoch           = m.num_epoch, 
      adj_parameter       = m.adj_parameter
  )
  # Parse label and choose output file to write
  model_file = f"{m.label}-model.pt"
  # Then save the trained model to disk
  save_model_dict(model_dict, model_file)
  # Also write the test input to file so that it is pass to downstream
  test_file = f"{m.label}-test_input.pkl"
  with open(test_file, 'wb') as f:
    pickle.dump(test_input, f)
  print(f"\nSaved input for testing at: {test_file}")
  write_metadata(m.data_folder, m.label)
  # And writing the metadata file for downstream
  
  
  return model_dict

In [36]:
import torch
import torch.nn.functional as F
# Custom imports from the package
from mogonet.utils import (
    one_hot_tensor, cal_sample_weight, 
    gen_adj_mat_tensor, gen_test_adj_mat_tensor, 
    cal_adj_mat_parameter, get_view_list, check_adj_param_size
)
from mogonet.prepare_trte_data import prepare_trte_data
from mogonet.models import init_model_dict, init_optim
import torch.nn as nn
import torch
import torch.nn.functional as F


In [19]:
cuda = True if torch.cuda.is_available() else False
if cuda:
    print("Found cuda, using GPU")
else:
    print("Cuda not found, using CPU")

# Parameters 
num_view = len(m.view_list)
dim_hvcdn = pow(m.num_class,num_view)
# ----------------------------------
# Modify LATER!!!!!!!
dim_he_list = [50] * num_view # Need a more robust way to decide this
# But it should be of length N (numbers of blocks), so might need to column number
# in each block minus some constant, such this number < column numebr of specific block
# ---------------------------------
data_tr_list, data_trte_list, trte_idx, labels_trte = prepare_trte_data(m.data_folder, m.view_list)
labels_tr_tensor = torch.LongTensor(labels_trte[trte_idx["tr"]])
onehot_labels_tr_tensor = one_hot_tensor(labels_tr_tensor, m.num_class)
sample_weight_tr = cal_sample_weight(labels_trte[trte_idx["tr"]], m.num_class)
sample_weight_tr = torch.FloatTensor(sample_weight_tr)

Found cuda, using GPU


In [25]:
print("This is tr list")
for i in data_tr_list:
    print(i.shape)
print("\nThis is trte list")
for j in data_trte_list:
    print(j.shape)

This is tr list
torch.Size([180, 204])
torch.Size([180, 260])

This is trte list
torch.Size([200, 204])
torch.Size([200, 260])


In [29]:
if cuda:
    labels_tr_tensor = labels_tr_tensor.cuda()
    onehot_labels_tr_tensor = onehot_labels_tr_tensor.cuda()
    sample_weight_tr = sample_weight_tr.cuda()

# TODO: FIX THIS AND MAKE IT MORE VERBOSE
# adj_parameter needs to be <= numples of rows in train data
adj_parameter = check_adj_param_size(m.adj_parameter, data_tr_list)
adj_tr_list, adj_te_list = gen_trte_adj_mat(data_tr_list, data_trte_list, trte_idx, adj_parameter)
# Feature dimension of each block in train data list
dim_list = [x.shape[1] for x in data_tr_list]

In [41]:
class GraphConvolution(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(out_features))
        nn.init.xavier_normal_(self.weight.data)
        if self.bias is not None:
            self.bias.data.fill_(0.0)
    
    def forward(self, x, adj):
        support = torch.mm(x, self.weight)
        output = torch.sparse.mm(adj, support)
        if self.bias is not None:
            return output + self.bias
        else:
            return output

In [40]:
def xavier_init(m):
    if type(m) == nn.Linear:
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
           m.bias.data.fill_(0.0)


class GCN_E(nn.Module):
    def __init__(self, in_dim, hgcn_dim, dropout):
        super().__init__()
        self.gc1 = GraphConvolution(in_dim, hgcn_dim[0])
        self.gc2 = GraphConvolution(hgcn_dim[0], hgcn_dim[1])
        self.gc3 = GraphConvolution(hgcn_dim[1], hgcn_dim[2])
        self.dropout = dropout

    def forward(self, x, adj):
        x = self.gc1(x, adj)
        x = F.leaky_relu(x, 0.25)
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.gc2(x, adj)
        x = F.leaky_relu(x, 0.25)
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.gc3(x, adj)
        x = F.leaky_relu(x, 0.25)
        
        return x

In [53]:
class GCN_E(nn.Module):
    def __init__(self, in_dim, hgcn_dim, dropout):
        super().__init__()
        self.gc1 = GraphConvolution(in_dim, hgcn_dim[0])
        self.gc2 = GraphConvolution(hgcn_dim[0], hgcn_dim[1])
        self.gc3 = GraphConvolution(hgcn_dim[1], hgcn_dim[2])
        self.dropout = dropout

    def forward(self, x, adj):
        x = self.gc1(x, adj)
        x = F.leaky_relu(x, 0.25)
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.gc2(x, adj)
        x = F.leaky_relu(x, 0.25)
        x = F.dropout(x, self.dropout, training=self.training)
        x = self.gc3(x, adj)
        x = F.leaky_relu(x, 0.25)
        
        return x

In [71]:
# Ignores extrat layers
GCN_E(dim_list[0],[50, 150, 200, 500], 0.5)

GCN_E(
  (gc1): GraphConvolution()
  (gc2): GraphConvolution()
  (gc3): GraphConvolution()
)

In [67]:
# Always assume this kind of input instead 
GCN_E(dim_list[0],[50, 150, 200], 0.5)

GCN_E(
  (gc1): GraphConvolution()
  (gc2): GraphConvolution()
  (gc3): GraphConvolution()
)

In [70]:
# Less than 3 omics, so always fail
GCN_E(dim_list[0], [50, 150], 0.5)

IndexError: list index out of range

In [72]:
print(dim_he_list)
model_dict = init_model_dict(num_view, m.num_class, dim_list, dim_he_list, dim_hvcdn)

[50, 50]


IndexError: list index out of range

In [ ]:
def init_model_dict(num_view, num_class, dim_list, dim_he_list, dim_hc, gcn_dopout=0.5):
    model_dict = {}
    for i in range(num_view):
        model_dict["E{:}".format(i+1)] = GCN_E(dim_list[i], dim_he_list, gcn_dopout)
        model_dict["C{:}".format(i+1)] = Classifier_1(dim_he_list[-1], num_class)
    if num_view >= 2:
        model_dict["C"] = VCDN(num_view, num_class, dim_hc)
    return model_dict

In [ ]:
# Initialize a model dictionary to update later
# Dim_he_list could be tuneable like [some_tune_num] * num_view or different numbers of each view
# like [k1, k2, k3, ... ] or just [k, k, k, ...], to list of length is equal to num_view
# model_dict = init_model_dict(num_view, num_class, dim_list, dim_he_list, dim_hvcdn)
# for m in model_dict:
#     if cuda:
#         model_dict[m].cuda()

# print("\nPretrain GCNs...")
# optim_dict = init_optim(num_view, model_dict, lr_e_pretrain, lr_c)
# # Some pretraining to happen
# if num_epoch_pretrain == 0:
#     print("Not pretraining")
# else:
#     for epoch in range(num_epoch_pretrain):
#         # Notice the model_dict is being changed under the hood
#         # for every model_dict[m].state_dict()
#         train_epoch(data_tr_list, adj_tr_list, labels_tr_tensor,
#                     onehot_labels_tr_tensor, sample_weight_tr, model_dict, 
#                     optim_dict, train_VCDN=False)
# Main logic to train now

# print("\nTraining...")
# optim_dict = init_optim(num_view, model_dict, lr_e, lr_c)
# for epoch in range(num_epoch+1):
#     # Notice the model_dict is being changed under the hood
#     # for every model_dict[m].state_dict()
#     train_epoch(data_tr_list, adj_tr_list, labels_tr_tensor,
#                 onehot_labels_tr_tensor, sample_weight_tr, model_dict, optim_dict)
# # Return relevant stuff back for test purposes
# test_input = {"data_list": data_trte_list,
#              "adj_list": adj_te_list,
#              "test_idxs": trte_idx["te"]
#              }
# return model_dict, test_input

In [13]:
mod_d = main(kl)

Found cuda, using GPU


/usr/local/lib/python3.10/dist-packages/mogonet/utils.py:79: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:618.)
  return sparse_tensortype(indices, values, x.size())


IndexError: list index out of range